In [29]:
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.models import load_model
import mediapipe.python.solutions.hands as mp_hands
import mediapipe.python.solutions.drawing_utils as drawing
import mediapipe.python.solutions.drawing_styles as drawing_styles

In [ ]:
model = load_model("rps_cnn.keras")
classes = ["Paper","Rock","Scissors"]
hands = mp_hands.Hands(
    static_image_mode=False,  # Set to False for processing video frames
    max_num_hands=2,           # Maximum number of hands to detect
    min_detection_confidence=0.5  # Minimum confidence threshold for hand detection
)

In [38]:
camera = cv2.VideoCapture(0)
while 1:
    global classes, hands
    success,frame = camera.read()
    if not success:
        break
    h,w,c = frame.shape
    pFrame = cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
    hands_detected = hands.process(frame)
    if hands_detected.multi_hand_landmarks:
            hand = hands_detected.multi_hand_landmarks[0]
            drawing.draw_landmarks(frame,hand,mp_hands.HAND_CONNECTIONS)
            x = [landmark.x for landmark in hand.landmark]
            y = [landmark.y for landmark in hand.landmark]
            center = np.array([np.mean(x)*w, np.mean(y)*h]).astype('int32')
            cv2.circle(frame, tuple(center), 10, (255,0,0), 1)  #for checking the center 
            cv2.rectangle(frame, (center[0]-100,center[1]-100), (center[0]+100,center[1]+100), (0,0,255), 1)
    pFrame = cv2.resize(pFrame,(128,128))
    pFrame = np.expand_dims(pFrame,axis=0)
    '''predict = model.predict(pFrame)
    predict = classes[np.argmax(predict[0])]
    cv2.putText(frame,f"{predict}",(100,100),cv2.FONT_HERSHEY_PLAIN,1,(0,0,200))'''
    cv2.imshow("Video",frame)
    if cv2.waitKey(1) == 27:
        break

camera.release()
cv2.destroyAllWindows()